# AMPS I · Week 09 — Lab
## PETSc SNES, line search and continuation

| | |
|---|---|
| **Lecture sections** | Part B (`NonlinearProblem`, reasons, boundary conditions, line search) · Part C (continuation, Bratu, custom Newton) · Part D (LLM) |
| **Time** | ≈ 70 min |
| **You will** | solve the Week 8 problem with `NonlinearProblem` and read SNES's diagnostics · see a failure that `solve()` does not report · compare line searches and initial guesses on a harder problem · run a continuation loop into a fold · pass the fold with a custom Newton solver and compare $\lambda^*$ with the literature |

**Reminder.** `%%fenicsx` cells run in a separate MPI process; ordinary cells have NumPy and matplotlib only. Data passes through `/tmp/fnx/`.

## ⚙️ Setup — run once per runtime
Installs DOLFINx 0.11 and registers the `%%fenicsx` cell magic. Re-run after every runtime restart.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FEniCSx on Google Colab — run once per runtime (≈ 3–5 min)
#  https://github.com/SeoulTechPSE/fenicsx-colab
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
import os, multiprocessing, subprocess
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/seoultechpse/fenicsx-colab.git'
REPO_DIR = Path('/content/fenicsx-colab')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# OpenMPI 5.x slot patch: Colab has 2 CPUs; allow up to 4 ranks (oversubscribed)
N_PROC = 4
with open('/root/hostfile', 'w') as f:
    f.write(f'localhost slots={N_PROC}\n')
os.environ['OMPI_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['PRTE_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['OMPI_MCA_osc'] = '^ucx'   # skip the UCX one-sided module (harmless start-up message)

DOLFINX_VERSION = '0.11'
USE_COMPLEX = False
USE_CLEAN   = False
opts = [f'--version {DOLFINX_VERSION}']
if USE_COMPLEX: opts.append('--complex')
if USE_CLEAN:   opts.append('--clean')
get_ipython().run_line_magic('run', f"{REPO_DIR / 'setup_fenicsx.py'} {' '.join(opts)}")

# Folder shared by %%fenicsx cells (computation) and ordinary cells (plotting)
os.makedirs('/tmp/fnx', exist_ok=True)

In [ ]:
%%fenicsx --info -np 2

## 1 · `NonlinearProblem` on the Week 8 problem  *(lecture §B1–B3)*
$-\nabla\cdot((1+u^2)\nabla u)=f$, $\tilde u=1+\sin(\pi x)y^2$, P1, $N=32$. The helper `snes_solve` builds a `NonlinearProblem`, records $\|F(u^k)\|$
with a monitor and returns the **converged reason**. Two initial guesses: $u^0=0$ everywhere, and $u^0=g$ on $\partial\Omega$ (0 inside).
Then the PETSc **defaults** (no `petsc_options`), read after a solve.

In [ ]:
%%fenicsx -np 2
from mpi4py import MPI
import numpy as np
import ufl
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import LinearProblem, NonlinearProblem

comm = MPI.COMM_WORLD
LU = {"ksp_type": "preonly", "pc_type": "lu"}
TIGHT = {**LU, "snes_rtol": 1e-10, "snes_atol": 1e-12, "snes_max_it": 50}

def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def boundary_dofs(msh, V):
    tdim = msh.topology.dim
    msh.topology.create_connectivity(tdim - 1, tdim)
    return fem.locate_dofs_topological(V, tdim - 1, mesh.exterior_facet_indices(msh.topology))

def norm_L2(msh, e):
    return np.sqrt(msh.comm.allreduce(fem.assemble_scalar(fem.form(e ** 2 * ufl.dx)), op=MPI.SUM))

_n = [0]
def snes_solve(F, uh, bcs, opts):
    """Solve F(uh; v) = 0 with PETSc SNES. Returns (reason, iterations, history of ||F||)."""
    _n[0] += 1
    problem = NonlinearProblem(F, uh, bcs=bcs, petsc_options_prefix=f"nls{_n[0]}_", petsc_options=opts)
    hist = []
    problem.solver.setMonitor(lambda snes, it, rnorm: hist.append(rnorm))
    problem.solve()
    return problem.solver.getConvergedReason(), problem.solver.getIterationNumber(), hist

q = lambda u: 1 + u ** 2

def week8_problem(N, k, A=1.0):
    """-div((1+u^2) grad u) = f on the unit square, u = g = u_ex on the boundary, u_ex = 1 + A sin(pi x) y^2 (A = 1: Week 8)."""
    msh = mesh.create_unit_square(comm, N, N)
    V = fem.functionspace(msh, ("Lagrange", k))
    x = ufl.SpatialCoordinate(msh)
    u_ex = 1 + A * ufl.sin(ufl.pi * x[0]) * x[1] ** 2
    f = -ufl.div(q(u_ex) * ufl.grad(u_ex))
    g = fem.Function(V); g.interpolate(fem.Expression(u_ex, V.element.interpolation_points))
    return msh, V, u_ex, f, fem.dirichletbc(g, boundary_dofs(msh, V))

def residual(uh, f):
    v = ufl.TestFunction(uh.function_space)
    return q(uh) * ufl.inner(ufl.grad(uh), ufl.grad(v)) * ufl.dx - f * v * ufl.dx

msh, V, u_ex, f, bc = week8_problem(32, 1)
for guess in ("zero", "zero+bc"):
    uh = fem.Function(V)
    if guess == "zero+bc":
        fem.set_bc(uh.x.array, [bc])                     # u0 = g on the boundary, 0 inside
    reason, its, hist = snes_solve(residual(uh, f), uh, [bc], TIGHT)
    err = norm_L2(msh, uh - u_ex)
    say(f"u0 = {guess:8s}: reason {reason}, {its} iterations, |F| = {' '.join(f'{r:.1e}' for r in hist)}, L2 error {err:.3e}")

uh = fem.Function(V)
problem = NonlinearProblem(residual(uh, f), uh, bcs=[bc], petsc_options_prefix="defaults_")
problem.solve()
s = problem.solver
say(f"defaults: {s.getType()}, line search {s.getLineSearch().getType()}, (rtol, atol, stol, max_it) = {s.getTolerances()}, "
    f"KSP {s.getKSP().getType()} + {s.getKSP().getPC().getType()}: {s.getIterationNumber()} Newton / {s.getLinearSolveIterations()} linear iterations")

> **✏️ Your turn (5 min).** (i) Solve again with `"snes_max_it": 2`. Does `solve()` complain? Print the reason and the $L^2$ error.
> (ii) Add `"snes_error_if_not_converged": True` and catch the exception with `try: ... except PETSc.Error as e:` (print `e.ierr`).
> Why is (i) dangerous inside a convergence study?

In [ ]:
%%fenicsx -np 2
# TODO: snes_max_it = 2, with and without snes_error_if_not_converged

## 2 · Line search and initial guess on a harder problem  *(lecture §A4, §B4)*
Ten times the Week 8 amplitude: $\tilde u=1+10\sin(\pi x)y^2$. Three initial guesses × three line searches. The histories go to
`/tmp/fnx/linesearch.npz`.

In [ ]:
%%fenicsx -np 2
from mpi4py import MPI
import numpy as np
import ufl
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import LinearProblem, NonlinearProblem

comm = MPI.COMM_WORLD
LU = {"ksp_type": "preonly", "pc_type": "lu"}
TIGHT = {**LU, "snes_rtol": 1e-10, "snes_atol": 1e-12, "snes_max_it": 50}

def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def boundary_dofs(msh, V):
    tdim = msh.topology.dim
    msh.topology.create_connectivity(tdim - 1, tdim)
    return fem.locate_dofs_topological(V, tdim - 1, mesh.exterior_facet_indices(msh.topology))

def norm_L2(msh, e):
    return np.sqrt(msh.comm.allreduce(fem.assemble_scalar(fem.form(e ** 2 * ufl.dx)), op=MPI.SUM))

_n = [0]
def snes_solve(F, uh, bcs, opts):
    """Solve F(uh; v) = 0 with PETSc SNES. Returns (reason, iterations, history of ||F||)."""
    _n[0] += 1
    problem = NonlinearProblem(F, uh, bcs=bcs, petsc_options_prefix=f"nls{_n[0]}_", petsc_options=opts)
    hist = []
    problem.solver.setMonitor(lambda snes, it, rnorm: hist.append(rnorm))
    problem.solve()
    return problem.solver.getConvergedReason(), problem.solver.getIterationNumber(), hist

q = lambda u: 1 + u ** 2

def week8_problem(N, k, A=1.0):
    """-div((1+u^2) grad u) = f on the unit square, u = g = u_ex on the boundary, u_ex = 1 + A sin(pi x) y^2 (A = 1: Week 8)."""
    msh = mesh.create_unit_square(comm, N, N)
    V = fem.functionspace(msh, ("Lagrange", k))
    x = ufl.SpatialCoordinate(msh)
    u_ex = 1 + A * ufl.sin(ufl.pi * x[0]) * x[1] ** 2
    f = -ufl.div(q(u_ex) * ufl.grad(u_ex))
    g = fem.Function(V); g.interpolate(fem.Expression(u_ex, V.element.interpolation_points))
    return msh, V, u_ex, f, fem.dirichletbc(g, boundary_dofs(msh, V))

def residual(uh, f):
    v = ufl.TestFunction(uh.function_space)
    return q(uh) * ufl.inner(ufl.grad(uh), ufl.grad(v)) * ufl.dx - f * v * ufl.dx

msh, V, u_ex, f, bc = week8_problem(32, 1, A=10.0)
v, w = ufl.TestFunction(V), ufl.TrialFunction(V)
u_lin = LinearProblem(ufl.inner(ufl.grad(w), ufl.grad(v)) * ufl.dx, f * v * ufl.dx, bcs=[bc],
                      petsc_options_prefix="lin_", petsc_options=LU).solve()          # the q = 1 solution
H = {}
for guess in ("zero", "zero+bc", "linear"):
    row = []
    for ls in ("basic", "bt", "l2"):
        uh = fem.Function(V)
        if guess == "zero+bc":
            fem.set_bc(uh.x.array, [bc])
        elif guess == "linear":
            uh.x.array[:] = u_lin.x.array
        reason, its, hist = snes_solve(residual(uh, f), uh, [bc], {**TIGHT, "snes_linesearch_type": ls})
        row.append(f"{ls:5s} reason {reason:3d}, {its:2d} its")
        H[f"{guess}|{ls}"] = np.array(hist)
    say(f"u0 = {guess:8s}   " + "   ".join(row))
if comm.rank == 0:
    np.savez("/tmp/fnx/linesearch.npz", **H)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

H = np.load("/tmp/fnx/linesearch.npz")
fig, ax = plt.subplots(figsize=(7, 4.2))
for key, style in (("zero|basic", "x--"), ("zero|bt", "s--"), ("zero+bc|bt", "o-"), ("zero|l2", "^-"), ("linear|basic", "d:")):
    ax.semilogy(H[key], style, label=key.replace("|", ", "))
ax.set_xlabel("Newton iteration"); ax.set_ylabel(r"$\|F(u^k)\|$"); ax.grid(alpha=0.3, which="both"); ax.legend(fontsize=9)
fig.tight_layout(); plt.show()

> **✏️ Your turn (10 min).** Continuation in the amplitude: make `A` a `fem.Constant`, build $\tilde u$, $f$ and the boundary function $g$ from it, create
> **one** `NonlinearProblem` with `"snes_linesearch_type": "basic"` (plain Newton), and solve for $A=2,4,6,8,10$ — updating `A.value` and
> re-interpolating $g$ before each `solve()`. Print the reason and the iterations per step and the final $L^2$ error. Compare the total with the table above.

In [ ]:
%%fenicsx -np 2
# TODO: continuation in A with plain Newton

## 3 · Bratu's problem: natural continuation into the fold  *(lecture §A2, §C1–C2)*
$-\Delta u=\lambda e^u$ on the unit square, $u=0$ on $\partial\Omega$ — the steady thermal-explosion (Frank-Kamenetskii) problem. P1, $N=32$.
One `NonlinearProblem` with $\lambda$ a `fem.Constant`; each solve starts from the previous solution. We keep a copy of the last converged state.

In [ ]:
%%fenicsx -np 2
from mpi4py import MPI
import numpy as np
import ufl
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import LinearProblem, NonlinearProblem

comm = MPI.COMM_WORLD
LU = {"ksp_type": "preonly", "pc_type": "lu"}
TIGHT = {**LU, "snes_rtol": 1e-10, "snes_atol": 1e-12, "snes_max_it": 50}

def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def boundary_dofs(msh, V):
    tdim = msh.topology.dim
    msh.topology.create_connectivity(tdim - 1, tdim)
    return fem.locate_dofs_topological(V, tdim - 1, mesh.exterior_facet_indices(msh.topology))

def norm_L2(msh, e):
    return np.sqrt(msh.comm.allreduce(fem.assemble_scalar(fem.form(e ** 2 * ufl.dx)), op=MPI.SUM))

_n = [0]
def snes_solve(F, uh, bcs, opts):
    """Solve F(uh; v) = 0 with PETSc SNES. Returns (reason, iterations, history of ||F||)."""
    _n[0] += 1
    problem = NonlinearProblem(F, uh, bcs=bcs, petsc_options_prefix=f"nls{_n[0]}_", petsc_options=opts)
    hist = []
    problem.solver.setMonitor(lambda snes, it, rnorm: hist.append(rnorm))
    problem.solve()
    return problem.solver.getConvergedReason(), problem.solver.getIterationNumber(), hist

msh = mesh.create_unit_square(comm, 32, 32)
V = fem.functionspace(msh, ("Lagrange", 1))
bc0 = fem.dirichletbc(0.0, boundary_dofs(msh, V), V)
lam = fem.Constant(msh, 0.0)
uh = fem.Function(V); v = ufl.TestFunction(V)
F = ufl.inner(ufl.grad(uh), ufl.grad(v)) * ufl.dx - lam * ufl.exp(uh) * v * ufl.dx
problem = NonlinearProblem(F, uh, bcs=[bc0], petsc_options_prefix="bratu_", petsc_options={**TIGHT, "snes_linesearch_type": "bt"})
umax = lambda: comm.allreduce(uh.x.array.max(), op=MPI.MAX)
rows = []
for L in (1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 6.5, 6.7, 6.8, 6.85):
    lam.value = L
    u_last = uh.x.array.copy()
    problem.solve()
    reason, its = problem.solver.getConvergedReason(), problem.solver.getIterationNumber()
    say(f"lam = {L:5.2f}: reason {reason:2d}, {its} iterations" + (f", max u = {umax():.4f}" if reason > 0 else ""))
    if reason <= 0:
        uh.x.array[:] = u_last                           # back to the last converged state
        break
    rows.append((L, umax()))
if comm.rank == 0:
    np.save("/tmp/fnx/natural.npy", np.array(rows))
for L in (6.8, 7.0):                                     # directly from u0 = 0
    uh.x.array[:] = 0.0; lam.value = L
    reason, its, hist = snes_solve(F, uh, [bc0], {**TIGHT, "snes_linesearch_type": "bt"})
    say(f"from u0 = 0, lam = {L}: reason {reason}, {its} iterations")

> **✏️ Your turn (5 min).** At $\lambda=2$ start SNES (`bt`) from $u^0=A\cdot16\,xy(1-x)(1-y)$ for $A=2,5,10$. Print the reason, the iterations and
> $\max u_h$. How many different solutions do you find? What does this say about "the" solution of a nonlinear problem?

In [ ]:
%%fenicsx -np 2
# TODO: lam = 2 from three initial guesses

## 4 · Passing the fold: a custom Newton solver  *(lecture §A5, §C3–C4)*
Unknowns $(u,\lambda)$ and the extra equation $\int_\Omega u\,dx=m$. `bratu_branch` steps in $m$ and solves the bordered system by block elimination:
$Ja=F$, $Jz=F_\lambda$ with the same LU factors, $\delta\lambda=(\ell\cdot a-c)/(\ell\cdot z)$, $\delta u=a-\delta\lambda z$. Read the code, then run it.

In [ ]:
%%fenicsx -np 2
from mpi4py import MPI
import numpy as np
import ufl
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import LinearProblem, NonlinearProblem

comm = MPI.COMM_WORLD
LU = {"ksp_type": "preonly", "pc_type": "lu"}
TIGHT = {**LU, "snes_rtol": 1e-10, "snes_atol": 1e-12, "snes_max_it": 50}

def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def boundary_dofs(msh, V):
    tdim = msh.topology.dim
    msh.topology.create_connectivity(tdim - 1, tdim)
    return fem.locate_dofs_topological(V, tdim - 1, mesh.exterior_facet_indices(msh.topology))

def norm_L2(msh, e):
    return np.sqrt(msh.comm.allreduce(fem.assemble_scalar(fem.form(e ** 2 * ufl.dx)), op=MPI.SUM))

_n = [0]
def snes_solve(F, uh, bcs, opts):
    """Solve F(uh; v) = 0 with PETSc SNES. Returns (reason, iterations, history of ||F||)."""
    _n[0] += 1
    problem = NonlinearProblem(F, uh, bcs=bcs, petsc_options_prefix=f"nls{_n[0]}_", petsc_options=opts)
    hist = []
    problem.solver.setMonitor(lambda snes, it, rnorm: hist.append(rnorm))
    problem.solve()
    return problem.solver.getConvergedReason(), problem.solver.getIterationNumber(), hist

from dolfinx.fem.petsc import apply_lifting, assemble_matrix, assemble_vector, create_matrix, create_vector, set_bc

def bratu_branch(msh, k, m_values, tol=1e-10, max_it=30):
    """-Lap u = lam e^u, u = 0 on the boundary. Unknowns (u, lam), constraint int u dx = m; one Newton solve per m.
    Bordered system [[J, F_lam], [ell^T, 0]] solved by block elimination. Returns rows (m, lam, max u, iterations)."""
    V = fem.functionspace(msh, ("Lagrange", k))
    bc0 = fem.dirichletbc(0.0, boundary_dofs(msh, V), V)
    uh, lam = fem.Function(V), fem.Constant(msh, 0.0)
    v, w = ufl.TestFunction(V), ufl.TrialFunction(V)
    F = ufl.inner(ufl.grad(uh), ufl.grad(v)) * ufl.dx - lam * ufl.exp(uh) * v * ufl.dx
    Fc, Jc = fem.form(F), fem.form(ufl.derivative(F, uh, w))
    Flc = fem.form(-ufl.exp(uh) * v * ufl.dx)                      # dF/dlam
    ell = assemble_vector(fem.form(v * ufl.dx))                    # d(int u dx)/du
    ell.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE); set_bc(ell, [bc0])
    mform = fem.form(uh * ufl.dx)
    A, b, bl = create_matrix(Jc), create_vector(V), create_vector(V)
    a, z = fem.Function(V), fem.Function(V)
    ksp = PETSc.KSP().create(msh.comm); ksp.setOperators(A); ksp.setType("preonly"); ksp.getPC().setType("lu")
    out = []
    for m in m_values:
        for it in range(1, max_it + 1):
            A.zeroEntries(); assemble_matrix(A, Jc, bcs=[bc0]); A.assemble()
            for vec, form in ((b, Fc), (bl, Flc)):
                with vec.localForm() as loc:
                    loc.set(0.0)
                assemble_vector(vec, form); apply_lifting(vec, [Jc], bcs=[[bc0]])
                vec.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE); set_bc(vec, [bc0])
            c = msh.comm.allreduce(fem.assemble_scalar(mform), op=MPI.SUM) - m
            ksp.solve(b, a.x.petsc_vec); ksp.solve(bl, z.x.petsc_vec)        # J a = F,  J z = dF/dlam (same LU factors)
            dlam = (ell.dot(a.x.petsc_vec) - c) / ell.dot(z.x.petsc_vec)     # collective dot products
            a.x.petsc_vec.axpy(-dlam, z.x.petsc_vec)                          # du = a - dlam z
            uh.x.petsc_vec.axpy(-1.0, a.x.petsc_vec); uh.x.scatter_forward()
            lam.value = float(lam.value) - dlam
            if abs(dlam) < tol and a.x.petsc_vec.norm() < tol:
                break
        else:
            raise RuntimeError(f"bordered Newton did not converge at m = {m}")
        out.append((m, float(lam.value), msh.comm.allreduce(uh.x.array.max(), op=MPI.MAX), it))
    ksp.destroy(); A.destroy(); b.destroy(); bl.destroy()
    return np.array(out)

def fold(out):
    """lam* and max u at the fold: vertex of the parabola through the three points with the largest lam."""
    i = int(np.argmax(out[:, 1]))
    p = np.polyfit(out[i - 1:i + 2, 0], out[i - 1:i + 2, 1], 2); ms = -p[1] / (2 * p[0])
    return np.polyval(p, ms), np.polyval(np.polyfit(out[i - 1:i + 2, 0], out[i - 1:i + 2, 2], 2), ms)

out = bratu_branch(mesh.create_unit_square(comm, 32, 32), 1, np.linspace(0.01, 1.6, 160))
ls, us = fold(out)
say(f"P1, N = 32: {len(out)} steps, at most {int(out[:, 3].max())} Newton iterations per step; fold lam* = {ls:.5f}, max u there {us:.4f}")
say(f"end of the branch: m = {out[-1, 0]:.2f}, lam = {out[-1, 1]:.3f}, max u = {out[-1, 2]:.3f}")
if comm.rank == 0:
    np.save("/tmp/fnx/branch.npy", out)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

br, nat = np.load("/tmp/fnx/branch.npy"), np.load("/tmp/fnx/natural.npy")
fig, ax = plt.subplots(figsize=(6.6, 4.3))
ax.plot(br[:, 1], br[:, 2], "-", lw=2, label="amplitude continuation (custom Newton)")
ax.plot(nat[:, 0], nat[:, 1], "o", label=r"natural continuation in $\lambda$ (SNES)")
ax.axvline(6.80812, color="0.5", ls=":", label=r"literature $\lambda^*=6.80812$")
ax.set_xlabel(r"$\lambda$"); ax.set_ylabel(r"$\max u_h$"); ax.grid(alpha=0.3); ax.legend(fontsize=9)
fig.tight_layout(); plt.show()

> **✏️ Your turn (10 min).** Compute $\lambda^*_h$ with `fold(bratu_branch(...))` for P1 on $N=16$ and $N=32$ (use `np.linspace(0.01, 1.2, 120)`), and for
> P2 on $N=16$. Print the errors against $\lambda^*=6.808124$ and the P1 error ratio. Which convergence order do you see, and why is it not $k+1$?

In [ ]:
%%fenicsx -np 2
# TODO: lam*_h for P1 (N = 16, 32) and P2 (N = 16)

## 5 · LLM-assisted coding: live demo  *(lecture Part D)*
Prompt (paste the API reference sheet first):

```text
Context: [API reference sheet]
Task: DOLFINx 0.11, one %%fenicsx -np 2 cell. Solve the Bratu problem -Lap u = lam exp(u) on the unit square, u = 0 on the boundary, P1, N = 32,
with dolfinx.fem.petsc.NonlinearProblem (PETSc SNES, newtonls, line search bt, LU). Use natural continuation lam = 1, 2, ..., 6, 6.5, 6.8
with the previous solution as the initial guess. Print lam, the SNES converged reason, the number of iterations and max(u) on rank 0.
```
Paste the result below **unchanged**, run it, then go through the checklist.

In [ ]:
%%fenicsx -np 2
# TODO: paste the %%fenicsx part your LLM produced (unchanged) and run it

**Verification checklist** — checks with a known answer

- [ ] $\lambda$ is a `fem.Constant` (a Python `float` is compiled into the form: every "step" solves the same problem — compare $\max u$ with §3)
- [ ] `petsc_options_prefix` is given; the linear solver is set explicitly (`preonly` + `lu`), not left at GMRES + block Jacobi
- [ ] The reason is checked after **every** solve; the loop stops or halves the step on failure
- [ ] The result of `solve()` is not unpacked as `(n, converged)` (0.10 API)
- [ ] $\max u$ agrees with §3 (0.078 at $\lambda=1$, 1.277 at $\lambda=6.8$); the same with `-np 1`